# Stroke — XAI extension

The reusable XAI framework is applied to the pre-specified class-balanced logistic-regression model using training-selected features. BMI imputation, feature selection and scaling are learned from the training data only.

The balanced model is specified for the extension independently of the held-out classifier ranking.

The framework generates explanations and assesses retraining-based stability, using ROC-AUC for scoring. Results are exported to `../../results/stroke`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

file_path = "../../data/stroke/healthcare-dataset-stroke-data.csv"


## Minimal model and data preparation

In [2]:
df = pd.read_csv(file_path)
df = df[df['gender'] != 'Other'].copy()
df['age'] = df['age'].astype(int)

y = df['stroke'].copy()
X_raw = df.drop(columns=['stroke', 'id']).copy()

# Split before imputation, encoding, feature selection, and scaling.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

for part in (X_train_raw, X_test_raw):
    part['bmi_missing'] = part['bmi'].isna().astype(int)

train_bmi_mean = X_train_raw['bmi'].mean()
X_train_raw['bmi'] = X_train_raw['bmi'].fillna(train_bmi_mean)
X_test_raw['bmi'] = X_test_raw['bmi'].fillna(train_bmi_mean)

categorical_cols = [
    'gender', 'smoking_status', 'Residence_type',
    'work_type', 'ever_married'
]
X_train_encoded = pd.get_dummies(
    X_train_raw, columns=categorical_cols, drop_first=False
)
X_test_encoded = pd.get_dummies(
    X_test_raw, columns=categorical_cols, drop_first=False
).reindex(columns=X_train_encoded.columns, fill_value=0)

# Learn the correlation-based feature set from training data only.
training_for_selection = X_train_encoded.copy()
training_for_selection['stroke'] = y_train.to_numpy()
training_correlations = training_for_selection.corr(numeric_only=True)['stroke'].abs()
selected_features = training_correlations[
    (training_correlations > 0.1)
    & (training_correlations.index != 'stroke')
].index.tolist()

X_train_full = X_train_encoded
X_test_full = X_test_encoded
X_train_sel = X_train_encoded[selected_features]
X_test_sel = X_test_encoded[selected_features]

In [3]:
scaler_full = StandardScaler()
X_train_full = pd.DataFrame(
    scaler_full.fit_transform(X_train_full),
    columns=X_train_encoded.columns, index=X_train_encoded.index
)
X_test_full = pd.DataFrame(
    scaler_full.transform(X_test_full),
    columns=X_train_encoded.columns, index=X_test_encoded.index
)

scaler_sel = StandardScaler()
X_train_sel = pd.DataFrame(
    scaler_sel.fit_transform(X_train_sel),
    columns=selected_features, index=X_train_encoded.index
)
X_test_sel = pd.DataFrame(
    scaler_sel.transform(X_test_sel),
    columns=selected_features, index=X_test_encoded.index
)

In [4]:
# Pre-specified extension model. Logistic regression provides a stable
# probabilistic baseline for the highly imbalanced outcome and was not
# selected by inspecting the held-out test ranking.
selected_model_name = "Logistic Regression"
final_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
final_model.fit(X_train_sel, y_train)
print("Pre-specified XAI model:", selected_model_name)

Pre-specified XAI model: Logistic Regression


In [5]:
from sklearn.base import clone

feature_names = list(X_train_sel.columns)

best_model = final_model


def train_model_fn(seed):
    model = clone(best_model)

    params = model.get_params(deep=False)

    if "random_state" in params:
        model.set_params(random_state=seed)

    model.fit(X_train_sel, y_train)

    return model

## Call the existing reusable XAI pipeline

In [6]:
import sys
from pathlib import Path

FRAMEWORK_DIR = Path.cwd().parent / "xai_framework"

if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
config = {
    "dataset_name": "stroke",

    "model": best_model,

    "X_train": X_train_sel,
    "X_test": X_test_sel,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": feature_names,

    "train_model_fn": train_model_fn,

    "scoring": "roc_auc",

    "output_dir": "../../results/stroke"
}

experiment = run_pipeline(config)

ExactExplainer explainer: 1023it [00:38, 26.62it/s]                          


Results exported to: ../../results/stroke


## XAI framework output sanity checks

In [8]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: stroke
Model: LogisticRegression
X_train shape: (4087, 7)
X_test shape: (1022, 7)
y_train length: 4087
y_test length: 1022
Feature names: 7


#### Consistency

In [9]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")

Basic shape validation: OK


#### Fitted model predicts on the configured data

In [10]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")

Probability range: 0.0088262240727006 0.9864089793069091
Predictions: (array([0, 1]), array([741, 281]))
Prediction test: OK


#### Results from Framework

In [11]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")

dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime'])
dict_keys(['raw', 'summary', 'rank_stability'])


#### Global XAI

In [12]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())

Native:


,Feature,Importance,RawValue,Method
0,age,1.674441,1.674441,coefficients
4,bmi_missing,0.331027,0.331027,coefficients
3,avg_glucose_level,0.212911,0.212911,coefficients
1,hypertension,0.205457,0.205457,coefficients
2,heart_disease,0.059397,0.059397,coefficients


Permutation:


,Feature,Importance,Std
0,age,0.291633,0.028744
3,avg_glucose_level,0.004798,0.003867
4,bmi_missing,0.003608,0.003121
2,heart_disease,0.001931,0.000932
5,ever_married_No,-0.000357,0.000490


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,age,0.224165,-0.019177
1,avg_glucose_level,0.028314,-0.004800
2,hypertension,0.021344,0.001532
3,bmi_missing,0.016160,0.002403
4,heart_disease,0.003113,0.000240


#### Feature coverage

In [13]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))

Native features: 7
Permutation features: 7
SHAP features: 7
Expected features: 7


#### Broken values

In [14]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


#### Rank agreement

In [15]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())

{'Native_vs_SHAP': np.float64(0.8928571428571429), 'Native_vs_Permutation': np.float64(0.7387687194103585), 'SHAP_vs_Permutation': np.float64(0.6306562238868914)}


,Feature,Importance_x,RawValue,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,age,1.674441,1.674441,coefficients,1.0,0.224165,-0.019177,1.0,0.291633,0.028744,1.0
1,bmi_missing,0.331027,0.331027,coefficients,2.0,0.016160,0.002403,4.0,0.003608,0.003121,3.0
2,avg_glucose_level,0.212911,0.212911,coefficients,3.0,0.028314,-0.004800,2.0,0.004798,0.003867,2.0
3,hypertension,0.205457,0.205457,coefficients,4.0,0.021344,0.001532,3.0,-0.000742,0.003242,7.0
4,heart_disease,0.059397,0.059397,coefficients,5.0,0.003113,0.000240,5.0,0.001931,0.000932,4.0


#### Representative cases

In [16]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)

,CaseType,y_true,y_pred,y_prob
207,TP,1,1,0.974578
662,TN,0,0,0.008826
96,FP,0,1,0.986409
553,FN,1,0,0.024162
361,Borderline,0,1,0.500372


#### LIME

In [17]:
lime_df = local_results["lime"]

print("LIME rows:", len(lime_df))
display(lime_df.head())

LIME rows: 35


,CaseIndex,Prediction,Probability,Rule,Weight
0,207,1,0.974578,age > 0.78,0.516819
1,207,1,0.974578,bmi_missing > -0.20,0.243195
2,207,1,0.974578,hypertension <= -0.32,-0.090770
3,207,1,0.974578,avg_glucose_level > 0.18,0.063422
4,207,1,0.974578,heart_disease > -0.25,0.040900


#### Stability

In [18]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])

,Feature,Importance,ImportanceStd,Seed,Rank
0,age,0.288523,0.039467,1,1.0
1,hypertension,-0.000790,0.002368,1,7.0
2,heart_disease,0.002794,0.000741,1,3.0
3,avg_glucose_level,0.003955,0.003595,1,2.0
4,bmi_missing,0.000181,0.004198,1,4.0


             Feature  MeanImportance  StdImportance  MeanRank   StdRank
0                age        0.286439       0.009285       1.0  0.000000
1  avg_glucose_level        0.005657       0.001279       2.0  0.000000
5      heart_disease        0.002392       0.000293       3.4  0.547723
2        bmi_missing        0.001752       0.002386       4.2  1.643168
3    ever_married_No       -0.000539       0.000165       5.5  0.000000
4   ever_married_Yes       -0.000539       0.000165       5.5  0.000000
6       hypertension       -0.001026       0.000381       6.4  1.341641


#### Exported folder

In [19]:
from pathlib import Path

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir)

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))

Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\stroke
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
lime_explanations.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [20]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi_missing', 'ever_married_No', 'ever_married_Yes', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,age,hypertension,heart_disease,avg_glucose_level,bmi_missing,ever_married_No,ever_married_Yes,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
207,1.533528,-0.323411,4.076319,2.550736,4.906488,-0.717905,0.717905,1,1,0.974578,0.474578,TP,207
662,-1.908375,-0.323411,-0.245319,-1.018439,-0.203812,1.392942,-1.392942,0,0,0.008826,0.491174,TN,662
96,1.533528,3.092042,4.076319,2.252783,4.906488,-0.717905,0.717905,0,1,0.986409,0.486409,FP,96
553,-1.290598,-0.323411,-0.245319,-1.073732,-0.203812,1.392942,-1.392942,1,0,0.024162,0.475838,FN,553
361,0.606862,-0.323411,-0.245319,1.870211,-0.203812,-0.717905,0.717905,0,1,0.500372,0.000372,Borderline,361


In [21]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("Stroke CaseIndex exists in memory.")

Stroke CaseIndex exists in memory.


In [22]:
stroke_cases = pd.read_csv(
    Path("../../results/stroke/representative_cases.csv")
)

print(stroke_cases.columns.tolist())
display(stroke_cases)

['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi_missing', 'ever_married_No', 'ever_married_Yes', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,age,hypertension,heart_disease,avg_glucose_level,bmi_missing,ever_married_No,ever_married_Yes,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
0,1.533528,-0.323411,4.076319,2.550736,4.906488,-0.717905,0.717905,1,1,0.974578,0.474578,TP,207
1,-1.908375,-0.323411,-0.245319,-1.018439,-0.203812,1.392942,-1.392942,0,0,0.008826,0.491174,TN,662
2,1.533528,3.092042,4.076319,2.252783,4.906488,-0.717905,0.717905,0,1,0.986409,0.486409,FP,96
3,-1.290598,-0.323411,-0.245319,-1.073732,-0.203812,1.392942,-1.392942,1,0,0.024162,0.475838,FN,553
4,0.606862,-0.323411,-0.245319,1.870211,-0.203812,-0.717905,0.717905,0,1,0.500372,0.000372,Borderline,361
